In [3]:
import torch
torch.manual_seed(42);
X=torch.randn(100,1)
y_true=2*X+1+0.1 * torch.randn(100,1)

w=torch.randn(1,requires_grad=True)
b=torch.randn(1,requires_grad=True)
learning_rate=0.1

for epoch in range(100):
  y_pred=X*w+b;
  loss=torch.mean((y_pred-y_true)**2)
  loss.backward()
  with torch.no_grad():
    w-=learning_rate*w.grad
    b-=learning_rate*b.grad

    w.grad.zero_()
    b.grad.zero_()
  if(epoch+1)%10==0:
   print(
            f"Epoch {epoch+1:03d} | Loss: {loss.item():.4f} | w: {w.item():.3f} | b: {b.item():.3f}"
        )

Epoch 010 | Loss: 0.0726 | w: 1.832 | b: 0.898
Epoch 020 | Loss: 0.0085 | w: 1.983 | b: 0.995
Epoch 030 | Loss: 0.0078 | w: 1.999 | b: 1.003
Epoch 040 | Loss: 0.0078 | w: 2.001 | b: 1.004
Epoch 050 | Loss: 0.0078 | w: 2.001 | b: 1.004
Epoch 060 | Loss: 0.0078 | w: 2.001 | b: 1.004
Epoch 070 | Loss: 0.0078 | w: 2.001 | b: 1.004
Epoch 080 | Loss: 0.0078 | w: 2.001 | b: 1.004
Epoch 090 | Loss: 0.0078 | w: 2.001 | b: 1.004
Epoch 100 | Loss: 0.0078 | w: 2.001 | b: 1.004


In [4]:
pip install torch

In [5]:
import torch
import torch.nn as nn
# class ke andar objjects /functions
class MNISTModel(nn.Module):
    def __init__(self, input_dim=784, hidden1=128, hidden2=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        # Flatten batch from (B, 1, 28, 28) or (B, 784) to (B, 784)
        if x.dim() > 2:
            x = x.view(x.size(0), -1)

        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        out = self.fc3(x)  # Raw logits output (no Softmax needed when using nn.CrossEntropyLoss)
        return out

# Instantiate model
model = MNISTModel()

# Test with dummy batch of 32 MNIST images (32, 1, 28, 28)
dummy_input = torch.randn(32, 1, 28, 28)
output = model(dummy_input)

# Verify parameter counts and dictionary structure
print(f"Output shape: {output.shape}")  # Expect: torch.Size([32, 10])
print("\n--- Model State Dict Keys & Shapes ---")
for param_tensor, value in model.state_dict().items():
    print(f"{param_tensor:15s}: {value.shape}")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params:,}")

Output shape: torch.Size([32, 10])

--- Model State Dict Keys & Shapes ---
fc1.weight     : torch.Size([128, 784])
fc1.bias       : torch.Size([128])
fc2.weight     : torch.Size([64, 128])
fc2.bias       : torch.Size([64])
fc3.weight     : torch.Size([10, 64])
fc3.bias       : torch.Size([10])

Total Trainable Parameters: 109,386


In [6]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Define transformations: Convert PIL image to Tensor & normalize to zero-mean, unit-variance
transform = transforms.Compose([
    transforms.ToTensor(), # Scales pixels from [0, 255] to [0.0, 1.0]
    transforms.Normalize((0.1307,), (0.3081,)) # MNIST global mean and std
])

# 2. Download and prepare datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 3. Wrap in DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 4. Inspect a single batch
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"Train Dataset Size: {len(train_dataset)} samples")
print(f"Test Dataset Size : {len(test_dataset)} samples")
print(f"Batch Images Shape: {images.shape}") # Expect: torch.Size([64, 1, 28, 28])
print(f"Batch Labels Shape: {labels.shape}") # Expect: torch.Size([64])

100%|██████████| 9.91M/9.91M [00:00<00:00, 43.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 892kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.72MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.09MB/s]

Train Dataset Size: 60000 samples
Test Dataset Size : 10000 samples
Batch Images Shape: torch.Size([64, 1, 28, 28])
Batch Labels Shape: torch.Size([64])


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 2. Instantiate Model, Loss, and Optimizer
model = MNISTModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Training & Validation Loop
epochs = 5

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    train_loss = running_loss / total_train
    train_acc = (correct_train / total_train) * 100

    # --- EVALUATION PHASE ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    val_loss = val_loss / total_val
    val_acc = (correct_val / total_val) * 100

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

Using device: cpu
Epoch 1/5 | Train Loss: 0.2674 | Train Acc: 92.10% | Val Loss: 0.1559 | Val Acc: 95.00%
Epoch 2/5 | Train Loss: 0.1148 | Train Acc: 96.44% | Val Loss: 0.0971 | Val Acc: 97.07%
Epoch 3/5 | Train Loss: 0.0798 | Train Acc: 97.51% | Val Loss: 0.0850 | Val Acc: 97.41%
Epoch 4/5 | Train Loss: 0.0628 | Train Acc: 97.92% | Val Loss: 0.0977 | Val Acc: 96.89%
Epoch 5/5 | Train Loss: 0.0486 | Train Acc: 98.41% | Val Loss: 0.0884 | Val Acc: 97.43%


In [8]:
import torch

# 1. Save Model State Dict
checkpoint_path = "mnist_mlp.pth"
torch.save(model.state_dict(), checkpoint_path)
print(f"Model weights saved to {checkpoint_path}")

# 2. Instantiate a fresh model (simulating a deployment environment)
fresh_model = MNISTModel().to(device)

# 3. Load saved parameters
fresh_model.load_state_dict(torch.load(checkpoint_path, map_location=device))
fresh_model.eval()  # Set to evaluation mode
print("Model loaded successfully for inference.")

# 4. Predict on a single image from the test set
sample_image, sample_label = test_dataset[0]  # Shape: (1, 28, 28)

# Add batch dimension (B=1, C=1, H=28, W=28) and move to device
input_tensor = sample_image.unsqueeze(0).to(device)

with torch.no_grad():
    logits = fresh_model(input_tensor)
    probabilities = torch.softmax(logits, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1).item()

print(f"True Label      : {sample_label}")
print(f"Predicted Class : {predicted_class}")
print(f"Confidence      : {probabilities[0][predicted_class].item() * 100:.2f}%")

Model weights saved to mnist_mlp.pth
Model loaded successfully for inference.
True Label      : 7
Predicted Class : 7
Confidence      : 100.00%
